[<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>](https://colab.research.google.com/github/ADRIANVM117/quant-research-notes/blob/main/volatility_target/01_rebalancing_policies/02_rebalancing_policies.ipynb)

Abra la notebook en Google Colab y seleccione **Run all**. La serie diaria validada de SPY se descarga automáticamente desde este repositorio; no se requieren claves de API, Google Drive ni ejecutar notebooks anteriores.

# Volatility targeting con bandas

## Nueva etapa — Del pronóstico al rebalanceo


Nuestra pregunta sigue siendo:

> ¿Pueden las bandas de no negociación reducir los costos de volatility targeting sin deteriorar significativamente el control del riesgo?

La notebook contendrá Dos bloques, desarrollados progresivamente:

| Bloque | Contenido                                              |
| ------ | ------------------------------------------------------ |
| 03     | Exposición objetivo y políticas P0, P1 y P2            |
| 04     | Backtest con ejecución en apertura $ (t+1) $ y costos  |




Recordemos las tres políticas:

| Política         | Regla                                                         |
| ---------------- | ------------------------------------------------------------- |
| P0 — Daily       | Rebalanceo diario hacia el peso objetivo                      |
| P1 — Banda 5 pp  | Rebalanceo cuando la desviación supera 5 puntos porcentuales  |
| P2 — Banda 10 pp | Rebalanceo cuando la desviación supera 10 puntos porcentuales |

Mantendremos inicialmente:

- $(\lambda=0.94)$.
- Volatilidad objetivo del 12% anual.
- Exposición máxima de 100%.
- Capital inicial de 200,000 MXN como escenario.
- Señal al cierre de $(t)$ y ejecución en apertura de $(t+1)$.
- Un benchmark de exposición constante.
- Costos explícitos de implementación.

Hay un problema interesante que debemos resolver antes de programar el backtest completo.

Si calculamos el peso objetivo al cierre, pero operamos en la apertura siguiente, el mercado puede modificar la exposición efectiva durante la noche.

Eso significa que nuestra política deberá distinguir:

1. El peso deseado calculado al cierre.
2. El peso efectivo antes de operar en la siguiente apertura.
3. La diferencia entre ambos.
4. La decisión de operar o permanecer dentro de la banda.

Y al combinar retornos ajustados por dividendos con precios de apertura sin ajustar, tendremos que contabilizar correctamente los dividendos y los eventos corporativos para no distorsionar el valor económico de la cartera.

Este problema ya no es únicamente estadístico: es un problema de diseño de una política implementable.


In [12]:
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlretrieve
import importlib.util
import subprocess
import sys

missing_packages = []
if importlib.util.find_spec('polars') is None:
    missing_packages.append('polars==1.44.2')
if importlib.util.find_spec('plotly') is None:
    missing_packages.append('plotly==6.2.0')
if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing_packages])

import polars as pl

SPY_URL = (
    'https://raw.githubusercontent.com/ADRIANVM117/quant-research-notes/'
    'main/volatility_target/series/SPY.parquet'
)

def find_local_spy() -> Path | None:
    for folder in (Path.cwd(), *Path.cwd().parents):
        for candidate in (folder, folder / 'volatility_target'):
            path = candidate / 'series' / 'SPY.parquet'
            if path.is_file():
                return path
    return None

spy_path = find_local_spy()
if spy_path is None:
    spy_path = Path.cwd() / 'series' / 'SPY.parquet'
    temporary_path = spy_path.with_suffix('.part')
    spy_path.parent.mkdir(parents=True, exist_ok=True)
    print('Descargando SPY.parquet desde GitHub...')
    try:
        urlretrieve(SPY_URL, temporary_path)
        temporary_path.replace(spy_path)
    except URLError as error:
        raise RuntimeError('No fue posible descargar SPY.parquet desde GitHub.') from error

spy = pl.read_parquet(spy_path)
required_columns = {'session_date', 'open', 'high', 'low', 'close', 'adjusted_close'}
if spy.is_empty() or required_columns.difference(spy.columns):
    raise ValueError('SPY.parquet no cumple el contrato esperado de datos diarios.')

print(f'Filas: {spy.height:,}')
print(f"Periodo: {spy['session_date'].min()} a {spy['session_date'].max()}")

Filas: 2,227
Periodo: 2018-01-02 a 2026-11-10


In [13]:
import math

LAMBDA = 0.94
WARMUP_RETURNS = 20
TRADING_DAYS = 252

# El retorno r_t está disponible al cierre de la fecha t.
returns = (
    spy.select(
        'session_date',
        pl.col('adjusted_close').pct_change().alias('adjusted_return'),
    )
    .drop_nulls()
)

return_dates = returns['session_date'].to_list()
return_values = returns['adjusted_return'].to_list()
forecast_variance = [None] * len(return_values)

# La primera varianza usa sólo los primeros 20 retornos disponibles.
forecast_variance[WARMUP_RETURNS - 1] = sum(
    value ** 2 for value in return_values[:WARMUP_RETURNS]
) / WARMUP_RETURNS

# Cada fila t emite el pronóstico para la siguiente sesión.
for index in range(WARMUP_RETURNS, len(return_values)):
    forecast_variance[index] = (
        LAMBDA * forecast_variance[index - 1]
        + (1 - LAMBDA) * return_values[index] ** 2
    )

ewma = (
    pl.DataFrame({
        'signal_date': return_dates,
        'forecast_for_date': return_dates[1:] + [None],
        'adjusted_return': return_values,
        'forecast_variance': forecast_variance,
    })
    .with_columns(
        pl.col('forecast_variance').sqrt().alias('forecast_vol_daily'),
        (pl.col('forecast_variance').sqrt() * math.sqrt(TRADING_DAYS)).alias(
            'forecast_vol_annualized'
        ),
    )
    .filter(pl.col('forecast_variance').is_not_null())
    .filter(pl.col('forecast_for_date').is_not_null())
)

print(f'Pronósticos EWMA disponibles: {ewma.height:,}')
ewma.head()

Pronósticos EWMA disponibles: 2,206


signal_date,forecast_for_date,adjusted_return,forecast_variance,forecast_vol_daily,forecast_vol_annualized
date,date,f64,f64,f64,f64
2018-01-31,2018-02-01,0.000497,0.000034,0.00587,0.093187
2018-02-01,2018-02-02,-0.001135,0.000032,0.005698,0.090455
2018-02-02,2018-02-05,-0.02177,0.000059,0.007678,0.12189
2018-02-05,2018-02-06,-0.041822,0.00016,0.012664,0.201028
2018-02-06,2018-02-07,0.019702,0.000174,0.013192,0.20942


## Paso 02 — Cronología operativa

Antes de programar una banda, fijamos qué ocurre en cada momento. La señal usa el cierre; la orden se ejecuta en la apertura siguiente. Por eso el peso que deseamos y el peso que realmente tenemos pueden ser distintos.

1. **Cierre de $(t)$.** Observamos el retorno, actualizamos EWMA y calculamos el peso deseado $(w^*_{t+1|t})$.
2. **Apertura de $(t+1)$, antes de operar.** El cambio entre cierre y apertura modifica el peso efectivo $(w^-_{t+1})$.
3. **Decisión.** P0 opera siempre; P1 y P2 operan sólo si $(|w^-_{t+1} - w^*_{t+1|t}|)$ supera su banda.
4. **Apertura de $(t+1)$, después de operar.** Si hay rebalanceo, el peso queda en el objetivo y se aplican los costos.
5. **Resto de la sesión.** La cartera queda expuesta al movimiento del mercado hasta el siguiente cierre.

Si $(q_t)$ son las acciones mantenidas y $(c_t)$ es el efectivo antes de operar, el peso efectivo en la apertura es:

$$
w^-_{t+1} = \frac{q_t O_{t+1}}{q_t O_{t+1} + c_t}.
$$

Esta distinción evita comparar la banda contra un peso antiguo. Dividendos y costos se incorporarán al construir el backtest; por ahora sólo hemos fijado la secuencia de decisiones.

In [14]:
# Precios observables en cada extremo de la decisión.
signal_closes = spy.select(
    pl.col('session_date').alias('signal_date'),
    pl.col('close').alias('signal_close'),
)
execution_opens = spy.select(
    pl.col('session_date').alias('forecast_for_date'),
    pl.col('open').alias('execution_open'),
)

decision_schedule = (
    ewma.select(
        'signal_date',
        'forecast_for_date',
        'forecast_vol_annualized',
    )
    .join(signal_closes, on='signal_date', how='inner')
    .join(execution_opens, on='forecast_for_date', how='inner')
    .with_columns(
        (pl.col('execution_open') / pl.col('signal_close') - 1).alias(
            'overnight_return'
        )
    )
    .sort('signal_date')
)

assert decision_schedule.select(
    (pl.col('forecast_for_date') > pl.col('signal_date')).all()
).item()

print(f'Pares señal-apertura disponibles: {decision_schedule.height:,}')
decision_schedule.head()

Pares señal-apertura disponibles: 2,206


signal_date,forecast_for_date,forecast_vol_annualized,signal_close,execution_open,overnight_return
date,date,f64,f64,f64,f64
2018-01-31,2018-02-01,0.093187,281.9,281.07,-0.002944
2018-02-01,2018-02-02,0.090455,281.58,280.08,-0.005327
2018-02-02,2018-02-05,0.12189,275.45,273.45,-0.007261
2018-02-05,2018-02-06,0.201028,263.93,259.94,-0.015118
2018-02-06,2018-02-07,0.20942,269.13,268.5,-0.002341


### Qué tenemos hasta aquí

Cada fila de `decision_schedule` conecta una señal EWMA calculada al cierre de \(t\) con la apertura ejecutable de \(t+1\).

El campo `overnight_return` mide el movimiento que ocurre entre ambos momentos. Todavía no es una ganancia o pérdida de cartera: sólo muestra por qué el peso deseado al cierre puede diferir del peso efectivo justo antes de operar.

Con esta cronología evitamos look-ahead y ya podemos construir el peso objetivo y las reglas P0, P1 y P2 sobre una secuencia implementable.

## Paso 03 — Peso objetivo

El pronóstico EWMA se convierte en una exposición deseada a SPY. Si el riesgo previsto sube, reducimos exposición; si baja, la aumentamos, sin superar 100%.

$$
w^*_{t+1|t} = \min\left(1, \frac{0.12}{\widehat{\sigma}_{t+1|t}}\right).
$$

El resto de la cartera permanece en efectivo. Por ejemplo, con una volatilidad pronosticada de 24%, el peso objetivo es 50%. La fórmula usa sólo la señal disponible al cierre de \(t\); todavía no decide si operamos ni calcula el peso efectivo.

In [15]:
TARGET_VOLATILITY = 0.12
MAX_EXPOSURE = 1.00

target_schedule = (
    decision_schedule
    .with_columns(
        pl.min_horizontal(
            pl.lit(MAX_EXPOSURE),
            pl.lit(TARGET_VOLATILITY) / pl.col('forecast_vol_annualized'),
        ).alias('target_weight')
    )
    .with_columns(
        (pl.lit(1.0) - pl.col('target_weight')).alias('cash_weight')
    )
)

assert target_schedule.select(
    ((pl.col('target_weight') >= 0) & (pl.col('target_weight') <= MAX_EXPOSURE)).all()
).item()

print(f"Peso objetivo medio: {target_schedule['target_weight'].mean():.1%}")
print(
    f"Sesiones con 100% en SPY: "
    f"{target_schedule.select((pl.col('target_weight') == MAX_EXPOSURE).mean()).item():.1%}"
)

target_schedule.select(
    'signal_date',
    'forecast_for_date',
    'forecast_vol_annualized',
    'target_weight',
    'cash_weight',
).head()

Peso objetivo medio: 79.9%
Sesiones con 100% en SPY: 34.5%


signal_date,forecast_for_date,forecast_vol_annualized,target_weight,cash_weight
date,date,f64,f64,f64
2018-01-31,2018-02-01,0.093187,1.0,0.0
2018-02-01,2018-02-02,0.090455,1.0,0.0
2018-02-02,2018-02-05,0.12189,0.984495,0.015505
2018-02-05,2018-02-06,0.201028,0.596931,0.403069
2018-02-06,2018-02-07,0.20942,0.57301,0.42699


### Interpretación

Con la muestra actual, el peso objetivo medio es cercano a 80% y la estrategia desea estar completamente expuesta a SPY en alrededor de un tercio de las sesiones. Cuando EWMA detecta más riesgo, el peso objetivo cae; su mínimo histórico es cercano a 15%.

Estos valores describen lo que la señal **quiere** mantener. Aún no representan la posición real de la cartera: el movimiento overnight, las bandas y los costos determinarán después si se llega a ese peso.

## Paso 04 — Estado de cartera y peso efectivo

Para saber si una banda exige operar, debemos conocer la cartera que realmente llega a la apertura. En cada sesión seguiremos cuatro variables: acciones de SPY, efectivo, valor total y peso efectivo.

$$
w^-_{t+1} = \frac{\text{acciones}_t \times O_{t+1}}{\text{valor de cartera en la apertura de } t+1}.
$$

Usaremos estas convenciones iniciales:

- Capital inicial: 200,000 unidades monetarias. Con acciones fraccionarias, la moneda no cambia los resultados porcentuales.
- Efectivo: rendimiento de 0% por ahora.
- Dividendos: se abonarán al efectivo mediante `dividend_amount`. `adjusted_close` se usa únicamente para la señal EWMA.

Todavía no elegimos una política. El siguiente código sólo construirá este estado y mostrará cómo el precio overnight puede alejar el peso efectivo del peso objetivo.

In [16]:
INITIAL_CAPITAL = 200_000.0

dividend_schedule = spy.select(
    pl.col('session_date').alias('forecast_for_date'),
    pl.col('dividend_amount').alias('dividend_per_share'),
)

portfolio_inputs = (
    target_schedule
    .join(dividend_schedule, on='forecast_for_date', how='inner')
    .sort('forecast_for_date')
)

def mark_to_open(shares, cash, open_price, dividend_per_share=0.0):
    """Valora una cartera justo antes de decidir si se rebalancea."""
    cash = cash + shares * dividend_per_share
    equity_value = shares * open_price
    portfolio_value = equity_value + cash
    if portfolio_value <= 0:
        raise ValueError('El valor de cartera debe ser positivo.')
    return {
        'cash': cash,
        'portfolio_value': portfolio_value,
        'effective_weight': equity_value / portfolio_value,
    }

# Elegimos la primera sesión con exposición parcial para observar el drift del peso.
first = (
    portfolio_inputs
    .with_row_index('input_index')
    .filter(pl.col('target_weight') < MAX_EXPOSURE)
    .row(0, named=True)
)
shares = INITIAL_CAPITAL * first['target_weight'] / first['execution_open']
cash = INITIAL_CAPITAL * first['cash_weight']
initial_state = mark_to_open(shares, cash, first['execution_open'])

# Si no se opera de nuevo, el siguiente precio de apertura altera el peso efectivo.
next_session = portfolio_inputs.row(first['input_index'] + 1, named=True)
next_state = mark_to_open(
    shares,
    cash,
    next_session['execution_open'],
    next_session['dividend_per_share'],
)

portfolio_state_example = pl.DataFrame([
    {
        'moment': 'apertura inicial, después de asignar el objetivo',
        'session_date': first['forecast_for_date'],
        'open_price': first['execution_open'],
        'target_weight': first['target_weight'],
        'effective_weight': initial_state['effective_weight'],
        'portfolio_value': initial_state['portfolio_value'],
    },
    {
        'moment': 'siguiente apertura, antes de decidir',
        'session_date': next_session['forecast_for_date'],
        'open_price': next_session['execution_open'],
        'target_weight': next_session['target_weight'],
        'effective_weight': next_state['effective_weight'],
        'portfolio_value': next_state['portfolio_value'],
    },
]).with_columns(
    (pl.col('effective_weight') - pl.col('target_weight')).alias('weight_deviation')
)

portfolio_state_example

moment,session_date,open_price,target_weight,effective_weight,portfolio_value,weight_deviation
str,date,f64,f64,f64,f64,f64
"""apertura inicial, después de a…",2018-02-05,273.45,0.984495,0.984495,200000.0,0.0
"""siguiente apertura, antes de d…",2018-02-06,259.94,0.596931,0.983703,190272.055573,0.386772


**Cómo leer el ejemplo.** En la primera fila, el peso efectivo coincide con el objetivo porque acabamos de asignar la cartera. En la segunda, no se ha operado: el cambio en la apertura y el nuevo pronóstico EWMA hacen que el peso efectivo (98.37%) quede muy por encima del objetivo (59.69%). Esa diferencia de 38.68 pp es la información que usará cada política para decidir si rebalancea.


## Paso 05 — Reglas de rebalanceo

Una banda no modifica el peso objetivo: sólo decide si vale la pena operar para alcanzarlo. Definimos la desviación como $(d = |w^- - w^*|)$.

- **P0 — Daily:** opera en cada apertura.
- **P1 — Banda 5 pp:** opera si $(d > 0.05)$.
- **P2 — Banda 10 pp:** opera si $(d > 0.10)$.

Cuando una política opera, llevará la cartera al peso $(w^*)$; cuando no opera, conserva sus acciones y efectivo. Usamos una desigualdad estricta: estar exactamente en el límite no activa una orden.


In [17]:
POLICIES = {
    'P0 — Daily': None,
    'P1 — Banda 5 pp': 0.05,
    'P2 — Banda 10 pp': 0.10,
}

def should_rebalance(effective_weight, target_weight, policy):
    """Indica si la política exige rebalancear en esta apertura."""
    if policy not in POLICIES:
        raise ValueError(f'Política no reconocida: {policy}')

    deviation = abs(effective_weight - target_weight)
    tolerance = POLICIES[policy]
    return True if tolerance is None else deviation > tolerance

# Comprobación pequeña de las tres reglas antes de integrarlas al backtest.
rule_examples = pl.DataFrame({
    'case': ['dentro de 5 pp', 'entre 5 y 10 pp', 'por encima de 10 pp'],
    'effective_weight': [0.76, 0.86, 0.92],
    'target_weight': [0.80, 0.80, 0.80],
}).with_columns(
    (pl.col('effective_weight') - pl.col('target_weight')).abs().alias('deviation')
)

for policy in POLICIES:
    rule_examples = rule_examples.with_columns(
        pl.struct(['effective_weight', 'target_weight']).map_elements(
            lambda row: should_rebalance(
                row['effective_weight'], row['target_weight'], policy
            ),
            return_dtype=pl.Boolean,
        ).alias(policy)
    )

rule_examples


case,effective_weight,target_weight,deviation,P0 — Daily,P1 — Banda 5 pp,P2 — Banda 10 pp
str,f64,f64,f64,bool,bool,bool
"""dentro de 5 pp""",0.76,0.8,0.04,true,false,false
"""entre 5 y 10 pp""",0.86,0.8,0.06,true,true,false
"""por encima de 10 pp""",0.92,0.8,0.12,true,true,true


## Paso 06 — Mecánica de una orden

Antes del backtest, definimos qué ocurre cuando una política decide operar. En la apertura, la cartera vale $(V^-)$ y la orden mueve la exposición de SPY al peso objetivo:

$$
q^+ = \frac{w^* V^-}{O}, \qquad c^+ = V^- - q^+ O.
$$

Aquí $(q^+)$ son las acciones después de operar, $(c^+)$ el efectivo y $(O)$ el precio de apertura. Por el momento no aplicamos costos: queremos comprobar sólo que la orden alcanza el objetivo. Los costos entrarán después, al recorrer todas las sesiones.


In [18]:
def rebalance_to_target(shares, cash, open_price, target_weight):
    """Rebalancea una cartera al objetivo, sin costos de transacción."""
    if not 0 <= target_weight <= MAX_EXPOSURE:
        raise ValueError('El peso objetivo está fuera de los límites permitidos.')

    portfolio_value = shares * open_price + cash
    target_equity = target_weight * portfolio_value
    target_shares = target_equity / open_price
    trade_notional = (target_shares - shares) * open_price
    cash_after = portfolio_value - target_equity

    return {
        'shares': target_shares,
        'cash': cash_after,
        'portfolio_value': portfolio_value,
        'trade_notional': trade_notional,
        'effective_weight': target_equity / portfolio_value,
    }

# Rebalanceamos la segunda fila del ejemplo anterior.
rebalanced_state = rebalance_to_target(
    shares=shares,
    cash=next_state['cash'],
    open_price=next_session['execution_open'],
    target_weight=next_session['target_weight'],
)

order_example = pl.DataFrame([
    {
        'moment': 'antes de operar',
        'effective_weight': next_state['effective_weight'],
        'cash': next_state['cash'],
        'portfolio_value': next_state['portfolio_value'],
        'trade_notional': 0.0,
    },
    {
        'moment': 'después de rebalancear',
        'effective_weight': rebalanced_state['effective_weight'],
        'cash': rebalanced_state['cash'],
        'portfolio_value': rebalanced_state['portfolio_value'],
        'trade_notional': rebalanced_state['trade_notional'],
    },
])

assert math.isclose(
    rebalanced_state['effective_weight'], next_session['target_weight']
)
order_example


moment,effective_weight,cash,portfolio_value,trade_notional
str,f64,f64,f64,f64
"""antes de operar""",0.983703,3100.932387,190272.055573,0.0
"""después de rebalancear""",0.596931,76692.810689,190272.055573,-73591.878302


**Cómo leer el ejemplo.** Antes de operar, SPY representa 98.37% de la cartera, mientras el objetivo es 59.69%. La orden vende aproximadamente 73,592 unidades monetarias de SPY —por eso `trade_notional` es negativo— y eleva el efectivo. El valor total no cambia porque aún no hay costos; sólo cambia la composición hasta alcanzar el objetivo.


## Paso 07 — Supuestos de implementación: aproximación a IBKR

A partir de aquí buscamos una simulación implementable, no una reproducción exacta de cada estado de cuenta. Los supuestos serán parámetros visibles para poder cambiarlos después.

- **Moneda.** SPY cotiza en USD. El motor conservará acciones, efectivo, dividendos y comisiones en USD. Los 200,000 MXN son el escenario de capital inicial, por lo que antes de activar costos fijos en USD habrá que convertirlos a USD con un tipo de cambio inicial. El efecto diario MXN/USD se añadirá como una capa posterior de reporte; no debemos mezclar MXN con precios de SPY directamente.
- **Plan y orden.** Supondremos una cuenta IBKR Pro Fixed, acciones fraccionarias y una orden Market-on-Open en la apertura siguiente. IBKR Lite está restringido a residentes de EE. UU. y Singapur.
- **Comisión.** Para cada orden de SPY usaremos `min(1% × notional_usd, max(USD 1.00, USD 0.005 × acciones))`. Este costo sustituye a la comisión plana de 5 bps: el mínimo por orden es especialmente importante para P0.
- **Deslizamiento.** Separaremos ejecución y comisión. El caso base supondrá 1 bp por lado: compras a `open × 1.0001` y ventas a `open × 0.9999`. Más adelante compararemos 0, 2 y 5 bps para comprobar que la conclusión no dependa de este supuesto.
- **IVA y otros impuestos.** No aplicaremos IVA al notional de SPY. El parámetro `iva_on_commission` quedará opcional y empezará en 0%; sólo se activaría si la entidad que factura la cuenta carga IVA sobre su comisión. Los impuestos personales sobre ganancias se mantienen fuera del backtest.
- **Dividendos y efectivo.** Los dividendos se acreditarán netos de una retención configurable; 10% será un escenario inicial sujeto a que el W-8BEN y la elegibilidad al tratado correspondan a la cuenta. El efectivo recibirá después una tasa histórica en USD consistente con las reglas de IBKR; mantenerlo permanentemente en 0% castigaría artificialmente las políticas que reducen exposición.

Las tarifas pueden cambiar y dependen de la entidad y plan de la cuenta. Referencias: [comisiones de acciones y ETF de IBKR](https://www.interactivebrokers.com/en/pricing/commissions-stocks.php), [costos de conversión FX de IBKR](https://www.interactivebrokers.com/en/pricing/commissions-spot-currencies.php), [intereses sobre efectivo de IBKR](https://www.interactivebrokers.com/en/accounts/fees/pricing-interest-rates.php) y [tratado fiscal México-EE. UU.](https://www.irs.gov/businesses/international-businesses/mexico-tax-treaty-documents).


In [19]:
from io import BytesIO
from urllib.error import URLError
from urllib.request import urlopen

INITIAL_CAPITAL_MXN = 200_000.0
FX_COMMISSION_BPS = 0.20
FX_MIN_COMMISSION_USD = 2.00

IBKR_COMMISSION_PER_SHARE_USD = 0.005
IBKR_MIN_COMMISSION_USD = 1.00
IBKR_MAX_COMMISSION_RATE = 0.01
SLIPPAGE_BPS = 1.0
IVA_ON_COMMISSION = 0.00
DIVIDEND_WITHHOLDING = 0.10

FRED_USDMXN_URL = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=DEXMXUS'

def load_mxn_per_usd():
    """Carga pesos mexicanos por dólar desde la serie pública DEXMXUS."""
    try:
        with urlopen(FRED_USDMXN_URL, timeout=30) as response:
            raw_fx = pl.read_csv(BytesIO(response.read()), null_values='.')
    except URLError as error:
        raise RuntimeError('No fue posible descargar DEXMXUS desde FRED.') from error

    date_column, rate_column = raw_fx.columns[:2]
    return (
        raw_fx
        .select(
            pl.col(date_column).cast(pl.Utf8).str.to_date('%Y-%m-%d').alias('session_date'),
            pl.col(rate_column).cast(pl.Float64, strict=False).alias('mxn_per_usd'),
        )
        .drop_nulls()
        .sort('session_date')
    )

fx_mxnusd = load_mxn_per_usd()
first_execution_date = portfolio_inputs.select(pl.col('forecast_for_date').min()).item()
initial_fx_row = (
    fx_mxnusd
    .filter(pl.col('session_date') <= first_execution_date)
    .tail(1)
)
if initial_fx_row.is_empty():
    raise ValueError('No hay tipo de cambio disponible para la fecha inicial.')

initial_mxn_per_usd = initial_fx_row.item(0, 'mxn_per_usd')
gross_initial_usd = INITIAL_CAPITAL_MXN / initial_mxn_per_usd
initial_fx_commission_usd = max(
    FX_MIN_COMMISSION_USD, gross_initial_usd * FX_COMMISSION_BPS / 10_000
)
INITIAL_CAPITAL_USD = gross_initial_usd - initial_fx_commission_usd

print(f'Inicio operativo: {first_execution_date}')
print(f'Tipo de cambio inicial: {initial_mxn_per_usd:.4f} MXN/USD')
print(f'Capital disponible: USD {INITIAL_CAPITAL_USD:,.2f}')


Inicio operativo: 2018-02-01
Tipo de cambio inicial: 18.3600 MXN/USD
Capital disponible: USD 10,891.25


In [20]:
def execution_price(open_price, trade_shares, slippage_bps=SLIPPAGE_BPS):
    """Aplica deslizamiento adverso al precio de apertura."""
    if trade_shares > 0:
        return open_price * (1 + slippage_bps / 10_000)
    if trade_shares < 0:
        return open_price * (1 - slippage_bps / 10_000)
    return open_price

def ibkr_stock_cost(trade_shares, fill_price, iva_rate=IVA_ON_COMMISSION):
    """Calcula comisión IBKR Pro Fixed e IVA opcional, ambos en USD."""
    shares = abs(trade_shares)
    notional = shares * fill_price
    if shares == 0:
        return {'notional': 0.0, 'commission': 0.0, 'iva': 0.0, 'total_cost': 0.0}

    commission = min(
        IBKR_MAX_COMMISSION_RATE * notional,
        max(IBKR_MIN_COMMISSION_USD, IBKR_COMMISSION_PER_SHARE_USD * shares),
    )
    iva = commission * iva_rate
    return {
        'notional': notional,
        'commission': commission,
        'iva': iva,
        'total_cost': commission + iva,
    }

# Ilustración con la venta necesaria en el ejemplo del Paso 06.
example_trade_shares = rebalanced_state['shares'] - shares
example_fill_price = execution_price(
    next_session['execution_open'], example_trade_shares
)
example_cost = ibkr_stock_cost(example_trade_shares, example_fill_price)

execution_cost_example = pl.DataFrame([
    {
        'side': 'sell' if example_trade_shares < 0 else 'buy',
        'shares': abs(example_trade_shares),
        'open_price': next_session['execution_open'],
        'fill_price': example_fill_price,
        **example_cost,
    }
])

assert example_fill_price < next_session['execution_open']
assert example_cost['total_cost'] >= IBKR_MIN_COMMISSION_USD
execution_cost_example


side,shares,open_price,fill_price,notional,commission,iva,total_cost
str,f64,f64,f64,f64,f64,f64,f64
"""sell""",283.111019,259.94,259.914006,73584.519114,1.415555,0.0,1.415555


**Cómo leer el ejemplo.** La orden es una venta, por lo que el precio de llenado queda ligeramente por debajo de la apertura debido al deslizamiento de 1 bp. La comisión se calcula aparte sobre las acciones negociadas; como `IVA_ON_COMMISSION = 0`, el IVA es cero. `total_cost` es el importe que el motor descontará del valor de cartera cuando ejecute esta orden.


## Paso 08 — Contrato del backtest

El motor recorrerá una sesión por vez y cada política recibirá exactamente la misma información, precios y supuestos. Su función no es predecir el mercado: sólo convierte una señal EWMA ya disponible en una cartera operable.

### Cronología de una sesión

| Momento | Información o acción |
| --- | --- |
| Cierre de `t-1` | Ya existe el pronóstico EWMA y el peso objetivo para `t`. No se opera aquí. |
| Apertura de `t` | Se acredita el dividendo programado, se valúan las acciones al precio `open` y se calcula el peso efectivo. |
| Decisión | P0 siempre opera; P1 y P2 sólo si la desviación supera su banda. La decisión usa el objetivo disponible desde el cierre anterior. |
| Ejecución | Si corresponde, la orden se llena alrededor de `open`, con deslizamiento adverso, comisión e IVA opcional. |
| Cierre de `t` | Las acciones se valúan con `close`; se registra el valor final de la cartera y se conserva la posición para la sesión siguiente. |

Así evitamos look-ahead: el cierre de `t` nunca interviene en una orden de la apertura de `t`.

### Estado y contabilidad

El estado mínimo será `shares`, `cash` y `portfolio_value`. Justo antes de decidir, el peso efectivo es:

$$
w^-_t = (shares × open_t) / (shares × open_t + cash).
$$

Los precios `open` y `close` sin ajustar sirven para valorar y ejecutar. `adjusted_close` sólo alimenta EWMA. En la fecha indicada por la serie, el dividendo bruto se convierte en efectivo neto: `shares × dividend_amount × (1 - dividend_withholding)`. Con esto no usamos precios ajustados para contar dos veces el dividendo.

Cuando hay una orden, el motor busca la cantidad de acciones que deje el peso posterior lo más cercano posible al objetivo, sin efectivo negativo ni apalancamiento. El costo reduce el valor de la cartera; por ello, rebalancear a 100% no significa comprar por todo el valor previo, sino comprar lo máximo compatible con pagar la orden.

### Costos y efectivo

El precio de ejecución será peor que `open` en 1 bp: las compras pagan más y las ventas reciben menos. Después se descuenta la comisión IBKR Pro Fixed y, sólo si se activa, el IVA sobre esa comisión. La conversión inicial MXN→USD ya descuenta su propia comisión.

El efectivo se mantiene en USD. Añadiremos una tasa histórica diaria de efectivo antes de interpretar resultados económicos; por ahora el motor dejará el punto de aplicación explícito. No simularemos liquidación T+1, impuestos personales sobre ganancias, préstamos de valores ni impacto de mercado dependiente del tamaño: son extensiones posibles, pero no deben ocultar la comparación central entre las bandas.

### Qué guardará cada política

Cada fila del registro incluirá fecha, objetivo, peso efectivo antes y después de operar, decisión, acciones, efectivo, valor a la apertura y al cierre, dividendo neto, precio de llenado, notional, comisión, IVA y costo total. A partir de este registro podremos calcular turnover, costos acumulados, volatilidad realizada y drawdown sin reconstruir la cartera después.

Antes de comparar rendimientos validaremos cuatro invariantes: no hay fechas con look-ahead, no hay efectivo negativo, los pesos permanecen entre 0% y 100%, y P0/P1/P2 cumplen exactamente sus reglas. Sólo entonces el backtest será una base válida para la evaluación de la notebook 03.


In [21]:
def rebalance_with_costs(shares, cash, open_price, target_weight):
    """Rebalancea al objetivo sin margen y descuenta ejecución y comisión."""
    if not 0 <= target_weight <= MAX_EXPOSURE:
        raise ValueError('El peso objetivo está fuera de los límites permitidos.')
    if shares < 0 or cash < 0 or open_price <= 0:
        raise ValueError('El estado inicial no es válido para una cartera long-only.')

    def state_for(shares_after):
        trade_shares = shares_after - shares
        fill_price = execution_price(open_price, trade_shares)
        costs = ibkr_stock_cost(trade_shares, fill_price)
        cash_after = cash - trade_shares * fill_price - costs['total_cost']
        portfolio_value = shares_after * open_price + cash_after
        effective_weight = (
            shares_after * open_price / portfolio_value if portfolio_value > 0 else 0.0
        )
        return {
            'shares': shares_after,
            'cash': cash_after,
            'portfolio_value': portfolio_value,
            'effective_weight': effective_weight,
            'trade_shares': trade_shares,
            'fill_price': fill_price,
            'trade_notional': trade_shares * fill_price,
            **costs,
        }

    current_value = shares * open_price + cash
    current_weight = shares * open_price / current_value if current_value > 0 else 0.0
    if math.isclose(current_weight, target_weight, abs_tol=1e-12):
        return state_for(shares)

    # La venta total da 0%; la compra máxima financiable deja efectivo en 0.
    lower_shares = 0.0
    upper_shares = shares
    if cash > 0:
        infeasible_upper = shares + cash / open_price
        for _ in range(80):
            midpoint = (upper_shares + infeasible_upper) / 2
            if state_for(midpoint)['cash'] >= 0:
                upper_shares = midpoint
            else:
                infeasible_upper = midpoint

    # Búsqueda binaria del número de acciones que alcanza el peso objetivo.
    for _ in range(80):
        midpoint = (lower_shares + upper_shares) / 2
        if state_for(midpoint)['effective_weight'] < target_weight:
            lower_shares = midpoint
        else:
            upper_shares = midpoint

    result = state_for((lower_shares + upper_shares) / 2)
    if result['cash'] < -1e-8:
        raise RuntimeError('La orden requeriría efectivo negativo.')
    return result

# Compra inicial: el costo reduce el valor, pero no permite margen.
initial_order = rebalance_with_costs(
    shares=0.0,
    cash=INITIAL_CAPITAL_USD,
    open_price=first['execution_open'],
    target_weight=first['target_weight'],
)

cost_aware_order_example = pl.DataFrame([
    {
        'portfolio_value_before': INITIAL_CAPITAL_USD,
        'target_weight': first['target_weight'],
        'effective_weight_after': initial_order['effective_weight'],
        'shares_after': initial_order['shares'],
        'cash_after': initial_order['cash'],
        'commission': initial_order['commission'],
        'total_cost': initial_order['total_cost'],
        'portfolio_value_after': initial_order['portfolio_value'],
    }
])

assert initial_order['cash'] >= -1e-8
assert math.isclose(
    initial_order['effective_weight'], first['target_weight'], abs_tol=1e-10
)
cost_aware_order_example


portfolio_value_before,target_weight,effective_weight_after,shares_after,cash_after,commission,total_cost,portfolio_value_after
f64,f64,f64,f64,f64,f64,f64,f64
10891.246187,0.984495,0.984495,39.204027,168.832964,1.0,1.0,10889.174153


**Cómo leer el ejemplo.** La cartera inicia en efectivo y compra SPY hasta alcanzar el peso objetivo. `effective_weight_after` coincide con el objetivo, mientras que `cash_after` permanece no negativo: no se usó margen. El valor posterior es ligeramente menor que el valor inicial porque ya incorpora el deslizamiento y la comisión. Esa diferencia es un costo de implementación, no un rendimiento de mercado.


## Paso 09 — Insumos diarios y rendimiento del efectivo

El motor necesita una fila por sesión con toda la información que puede usar en esa fecha: objetivo calculado previamente, `open`, `close`, dividendo y tasa del efectivo. `open` permite ejecutar; `close` permite valorar al terminar la sesión; el dividendo y el interés cambian el efectivo disponible.

Usaremos la tasa efectiva de fondos federales de FRED (`DFF`) como referencia histórica diaria para el efectivo en USD. No pretende reconstruir exactamente cada cambio contractual de IBKR; aproxima una cuenta IBKR Pro con reglas visibles:

- Los primeros USD 10,000 de efectivo no generan interés.
- Sobre el excedente, la tasa anual será `max(0, DFF - 0.50 pp)`.
- Si el NAV es menor a USD 100,000, la tasa se multiplica por `NAV / 100,000`, con máximo de 1.

La tasa se aplicará sólo al efectivo que permaneció durante la noche, con base en la última observación disponible antes de la sesión. En forma diaria:

$$
cash \ after \ interest = cash \ before \ interest × (1 + cash \ rate \ annual / 360).
$$

Para un capital inicial cercano a USD 10,900, el interés probablemente será pequeño o nulo la mayor parte del tiempo por el umbral de USD 10,000. Aun así, incluirlo evita que el motor dependa de una suposición implícita y permite reutilizarlo para carteras mayores.

El dividendo neto se acreditará antes de decidir el rebalanceo. El rendimiento del efectivo no participa en EWMA ni modifica el peso objetivo: sólo afecta la contabilidad económica de la cartera. La serie DFF es pública y se descargará automáticamente desde FRED, igual que el tipo de cambio inicial.


In [22]:
IBKR_CASH_THRESHOLD_USD = 10_000.0
IBKR_CASH_FULL_NAV_USD = 100_000.0
IBKR_CASH_SPREAD = 0.005
FRED_DFF_URL = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=DFF'

def load_fred_dff():
    """Carga DFF como tasa anual porcentual."""
    try:
        with urlopen(FRED_DFF_URL, timeout=30) as response:
            raw_dff = pl.read_csv(BytesIO(response.read()), null_values='.')
    except URLError as error:
        raise RuntimeError('No fue posible descargar DFF desde FRED.') from error

    date_column, rate_column = raw_dff.columns[:2]
    return (
        raw_dff
        .select(
            pl.col(date_column).cast(pl.Utf8).str.to_date('%Y-%m-%d').alias('session_date'),
            pl.col(rate_column).cast(pl.Float64, strict=False).alias('dff_percent'),
        )
        .drop_nulls()
        .sort('session_date')
    )

dff = load_fred_dff()

# El motor verá la tasa de la sesión previa, nunca la observación contemporánea.
dff_for_sessions = (
    portfolio_inputs
    .select('forecast_for_date')
    .sort('forecast_for_date')
    .join_asof(
        dff,
        left_on='forecast_for_date',
        right_on='session_date',
        strategy='backward',
    )
    .with_columns(
        pl.col('dff_percent').shift(1).fill_null(0.0).alias('dff_percent_available')
    )
    .select('forecast_for_date', 'dff_percent_available')
)

closing_prices = spy.select(
    pl.col('session_date').alias('forecast_for_date'),
    pl.col('close').alias('close_price'),
)

backtest_inputs = (
    portfolio_inputs
    .join(closing_prices, on='forecast_for_date', how='inner')
    .join(dff_for_sessions, on='forecast_for_date', how='inner')
    .sort('forecast_for_date')
)

required_backtest_columns = {
    'signal_date', 'forecast_for_date', 'execution_open', 'close_price',
    'target_weight', 'dividend_per_share', 'dff_percent_available',
}
if required_backtest_columns.difference(backtest_inputs.columns):
    raise ValueError('Faltan columnas necesarias para el backtest.')
if backtest_inputs.null_count().select(pl.sum_horizontal(pl.all())).item() != 0:
    raise ValueError('La tabla diaria contiene valores faltantes.')

print(f'Sesiones listas para el motor: {backtest_inputs.height:,}')
print(f'Periodo: {backtest_inputs["forecast_for_date"].min()} a {backtest_inputs["forecast_for_date"].max()}')
backtest_inputs.head(3)


Sesiones listas para el motor: 2,206
Periodo: 2018-02-01 a 2026-11-10


signal_date,forecast_for_date,forecast_vol_annualized,signal_close,execution_open,overnight_return,target_weight,cash_weight,dividend_per_share,close_price,dff_percent_available
date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64
2018-01-31,2018-02-01,0.093187,281.9,281.07,-0.002944,1.0,0.0,0.0,281.58,0.0
2018-02-01,2018-02-02,0.090455,281.58,280.08,-0.005327,1.0,0.0,0.0,275.45,1.42
2018-02-02,2018-02-05,0.12189,275.45,273.45,-0.007261,0.984495,0.015505,0.0,263.93,1.42


**Cómo leer el resultado.** Las 2,206 filas representan todas las sesiones en las que existe una señal EWMA ejecutable y los datos necesarios para contabilizar la cartera. La validación confirma que no hay huecos ni fechas adelantadas: cada objetivo proviene de un cierre anterior. Esta tabla todavía no contiene rendimientos de P0, P1 o P2; es el insumo común que recibirá cada política.


In [23]:
def cash_interest(cash, portfolio_value, dff_percent):
    """Calcula un día de interés sobre el efectivo elegible de IBKR."""
    eligible_cash = max(0.0, cash - IBKR_CASH_THRESHOLD_USD)
    nav_scale = min(1.0, portfolio_value / IBKR_CASH_FULL_NAV_USD)
    annual_rate = max(0.0, dff_percent / 100 - IBKR_CASH_SPREAD) * nav_scale
    daily_interest = eligible_cash * annual_rate / 360
    return {
        'eligible_cash': eligible_cash,
        'cash_rate_annual': annual_rate,
        'daily_interest': daily_interest,
    }

example_dff = backtest_inputs['dff_percent_available'][1]
cash_interest_example = pl.DataFrame([
    {
        'case': 'capital inicial',
        'cash': INITIAL_CAPITAL_USD,
        'portfolio_value': INITIAL_CAPITAL_USD,
        'dff_percent': example_dff,
        **cash_interest(INITIAL_CAPITAL_USD, INITIAL_CAPITAL_USD, example_dff),
    },
    {
        'case': 'NAV de USD 100,000',
        'cash': 20_000.0,
        'portfolio_value': 100_000.0,
        'dff_percent': example_dff,
        **cash_interest(20_000.0, 100_000.0, example_dff),
    },
])

assert cash_interest(5_000.0, 20_000.0, example_dff)['daily_interest'] == 0.0
cash_interest_example


case,cash,portfolio_value,dff_percent,eligible_cash,cash_rate_annual,daily_interest
str,f64,f64,f64,f64,f64,f64
"""capital inicial""",10891.246187,10891.246187,1.42,891.246187,0.001002,0.002481
"""NAV de USD 100,000""",20000.0,100000.0,1.42,10000.0,0.0092,0.255556


## Paso 10 — Motor diario de las políticas

Ahora uniremos todas las piezas en una función reutilizable, `run_policy`. Recibirá `backtest_inputs` y el nombre de una política; devolverá una fila por sesión con el estado completo de la cartera. Las tres políticas recorrerán exactamente las mismas 2,206 sesiones.

La cartera inicia antes de la primera apertura con cero acciones y el capital inicial ya convertido a USD. Para cada sesión, el motor seguirá este orden:

1. Acreditar el interés correspondiente al efectivo que sobrevivió desde el cierre anterior. La primera sesión no recibe interés previo.
2. Acreditar el dividendo neto de las acciones que se tenían antes de abrir.
3. Valorar la posición con `execution_open` y calcular el peso efectivo antes de operar.
4. Consultar `should_rebalance` con el objetivo disponible desde el cierre anterior.
5. Si la política lo solicita, llamar a `rebalance_with_costs`; si no, conservar acciones y efectivo.
6. Valorar la misma cartera al `close_price` y guardar el estado para la siguiente sesión.

El interés se acredita antes del dividendo y de la orden porque remunera el efectivo de la noche anterior. El dividendo acreditado en esta apertura comenzará a devengar interés hasta el siguiente periodo. Es una simplificación explícita de la liquidación real.

Cada fila registrará dos pesos: `effective_weight_before` para evaluar la banda y `effective_weight_after` para confirmar el resultado de la orden. También separará `rebalance_requested` de `trade_executed`: P0 puede pedir revisión diaria aunque, de forma excepcional, no necesite cambiar acciones.

El valor económico al final de la sesión será:

$$
portfolio \ value \ close = shares \ after × close \ price + cash \ after.
$$

Por tanto, los retornos posteriores incorporarán movimiento intradía, dividendos netos, interés y costos de implementación. Antes de interpretar cualquier métrica, el código comprobará que no haya efectivo negativo, acciones negativas, pesos fuera de 0%–100% ni objetivos fuera de la cronología permitida.


In [24]:
def hold_position(shares, cash, open_price):
    """Devuelve el estado de apertura cuando no se envía una orden."""
    portfolio_value = shares * open_price + cash
    return {
        'shares': shares,
        'cash': cash,
        'portfolio_value': portfolio_value,
        'effective_weight': shares * open_price / portfolio_value,
        'trade_shares': 0.0,
        'fill_price': None,
        'trade_notional': 0.0,
        'notional': 0.0,
        'commission': 0.0,
        'iva': 0.0,
        'total_cost': 0.0,
    }

def run_policy(inputs, policy):
    """Ejecuta una política de rebalanceo sobre toda la tabla diaria."""
    if policy not in POLICIES:
        raise ValueError(f'Política no reconocida: {policy}')

    shares = 0.0
    cash = INITIAL_CAPITAL_USD
    records = []

    for session_index, row in enumerate(inputs.iter_rows(named=True)):
        open_price = row['execution_open']

        # Interés del efectivo que venía de la sesión anterior.
        opening_value_before_interest = shares * open_price + cash
        interest = (
            cash_interest(cash, opening_value_before_interest, row['dff_percent_available'])
            if session_index > 0 else {'daily_interest': 0.0}
        )
        cash += interest['daily_interest']

        dividend_net = shares * row['dividend_per_share'] * (1 - DIVIDEND_WITHHOLDING)
        cash += dividend_net

        value_before_trade = shares * open_price + cash
        effective_weight_before = (
            shares * open_price / value_before_trade if value_before_trade > 0 else 0.0
        )
        rebalance_requested = should_rebalance(
            effective_weight_before, row['target_weight'], policy
        )

        if rebalance_requested:
            state = rebalance_with_costs(
                shares, cash, open_price, row['target_weight']
            )
        else:
            state = hold_position(shares, cash, open_price)

        trade_executed = abs(state['trade_shares']) > 1e-10
        shares = state['shares']
        cash = state['cash']
        portfolio_value_close = shares * row['close_price'] + cash

        records.append({
            'policy': policy,
            'session_date': row['forecast_for_date'],
            'signal_date': row['signal_date'],
            'target_weight': row['target_weight'],
            'effective_weight_before': effective_weight_before,
            'effective_weight_after': state['effective_weight'],
            'rebalance_requested': rebalance_requested,
            'trade_executed': trade_executed,
            'shares': shares,
            'cash': cash,
            'dividend_net': dividend_net,
            'cash_interest': interest['daily_interest'],
            'open_price': open_price,
            'fill_price': state['fill_price'] if trade_executed else None,
            'close_price': row['close_price'],
            'trade_shares': state['trade_shares'],
            'trade_notional': state['trade_notional'],
            'turnover_notional': abs(state['trade_notional']),
            'commission': state['commission'],
            'iva': state['iva'],
            'total_cost': state['total_cost'],
            'portfolio_value_open': state['portfolio_value'],
            'portfolio_value_close': portfolio_value_close,
        })

    history = pl.DataFrame(records)
    if history.height != inputs.height:
        raise RuntimeError('El motor no recorrió todas las sesiones.')
    if not (history['session_date'] > history['signal_date']).all():
        raise RuntimeError('Se detectó una violación de cronología.')
    if not (history['cash'] >= -1e-8).all() or not (history['shares'] >= 0).all():
        raise RuntimeError('Se detectó una posición no permitida.')
    if not history['effective_weight_after'].is_between(0, MAX_EXPOSURE).all():
        raise RuntimeError('Se detectó un peso fuera de límites.')
    return history

policy_histories = {policy: run_policy(backtest_inputs, policy) for policy in POLICIES}

# Validamos que cada regla se aplicó como fue definida.
assert policy_histories['P0 — Daily']['rebalance_requested'].all()
for policy, tolerance in POLICIES.items():
    if tolerance is not None:
        history = policy_histories[policy]
        expected = (history['effective_weight_before'] - history['target_weight']).abs() > tolerance
        assert (history['rebalance_requested'] == expected).all()

policy_operational_summary = pl.concat([
    history.select(
        pl.lit(policy).alias('policy'),
        pl.len().alias('sessions'),
        pl.col('trade_executed').sum().alias('trades'),
        pl.col('turnover_notional').sum().alias('turnover_notional'),
        pl.col('total_cost').sum().alias('total_cost'),
        pl.col('portfolio_value_close').last().alias('final_portfolio_value'),
    )
    for policy, history in policy_histories.items()
])

policy_operational_summary


policy,sessions,trades,turnover_notional,total_cost,final_portfolio_value
str,u32,u32,f64,f64,f64
"""P0 — Daily""",2206,1510,483265.200112,1346.30249,20975.008852
"""P1 — Banda 5 pp""",2206,266,302501.777276,266.0,22206.832157
"""P2 — Banda 10 pp""",2206,109,214589.184892,109.0,22008.203514


**Cómo leer el resultado.** La tabla resume la implementación, no una conclusión de desempeño todavía. `trades` cuenta las órdenes que realmente modificaron acciones; `turnover_notional` acumula el valor absoluto negociado; `total_cost` suma comisión e IVA; y `final_portfolio_value` es el valor al cierre final, después de precios, dividendos, interés y costos. Las validaciones internas ya confirmaron que las tres políticas recorrieron las mismas sesiones sin look-ahead, efectivo negativo ni pesos fuera del intervalo permitido.


## Paso 11 — Resultados operativos

Antes de medir rendimiento y riesgo, observaremos cómo se comporta cada política como proceso de implementación. Estas salidas responden si las bandas realmente reducen actividad y fricción, y a qué costo de alejarse temporalmente del objetivo.

Mostraremos cuatro elementos:

| Resultado | Qué permite observar |
| --- | --- |
| Peso objetivo vs. peso efectivo | Cuándo la cartera sigue el objetivo y cuándo una banda tolera desviaciones. |
| Operaciones por política | Con qué frecuencia P0, P1 y P2 modifican realmente la posición. |
| Turnover acumulado | Cuánto capital se negocia a lo largo del tiempo, independiente del rendimiento. |
| Costos acumulados | Cuánto valor se pierde por comisión e IVA bajo cada regla. |

La serie de pesos se mostrará en una ventana reciente para que pueda leerse; las demás métricas usarán toda la muestra. Estas gráficas describen la operación del motor y no prueban todavía que una política sea mejor: la respuesta económica y estadística se desarrollará en la notebook 03.
